# DL Volatility Forecasting: LSTM vs. TFT vs. GARCH Baseline

This notebook walks through the full pipeline:
1. Data download & feature engineering
2. Dataset exploration
3. LSTM training & evaluation
4. Temporal Fusion Transformer training & attention analysis
5. GARCH / XGBoost baselines
6. Model comparison: QLIKE, RMSE-log-vol, Diebold-Mariano
7. Regime-conditional performance

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import seaborn as sns
import torch

## 1. Data Download & Feature Engineering

In [ ]:
from data.features import download_and_engineer, FEATURE_COLS

df = download_and_engineer()
print(f"Dataset: {df.index[0].date()} → {df.index[-1].date()}  ({len(df)} rows)")
df.head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

realized_vol = np.sqrt(df['target']) * np.sqrt(252)
axes[0].plot(df.index, realized_vol, linewidth=0.8, color='navy')
axes[0].set_ylabel('Next-day Realized Vol (ann.)')
axes[0].set_title('SPY Volatility Landscape 2010–present')

axes[1].plot(df.index, df['vix'], linewidth=0.8, color='firebrick', label='VIX')
axes[1].axhline(20, linestyle='--', color='grey', linewidth=0.8)
axes[1].axhline(25, linestyle='--', color='orange', linewidth=0.8)
axes[1].set_ylabel('VIX')

axes[2].plot(df.index, df['log_ret'], linewidth=0.5, color='green', alpha=0.7)
axes[2].set_ylabel('Log Return')

plt.tight_layout()
plt.show()

In [ ]:
from data.dataset import get_splits
train, val, test = get_splits(df)
print(f"Train: {train.index[0].date()} → {train.index[-1].date()}  ({len(train)} days)")
print(f"Val:   {val.index[0].date()} → {val.index[-1].date()}  ({len(val)} days)")
print(f"Test:  {test.index[0].date()} → {test.index[-1].date()}  ({len(test)} days)")

## 2. LSTM Training

In [ ]:
from training.train_lstm import train_lstm
model, train_loader, val_loader, test_loader, scaler, df_full = train_lstm(
    epochs=150, patience=15, batch_size=64
)

In [ ]:
import pandas as pd
log = pd.read_csv('../outputs/training_log_lstm.csv')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(log['epoch'], log['train_loss'], label='Train QLIKE')
ax.plot(log['epoch'], log['val_loss'], label='Val QLIKE')
ax.set_xlabel('Epoch')
ax.set_ylabel('QLIKE')
ax.set_title('LSTM Training Curves')
ax.legend()
plt.tight_layout()
plt.show()

## 3. TFT Training

In [ ]:
from training.train_tft import train_tft
tft, training_ds, val_ds, scaled_df, train_mask, val_mask, test_mask, tft_ckpt = train_tft(
    max_epochs=100, patience=15, batch_size=64
)
print(f'Best TFT checkpoint: {tft_ckpt}')

## 4. Baselines (GARCH + XGBoost)

In [ ]:
from models.baseline import BaselineForecaster
bf = BaselineForecaster()
bf.fit(df)
print('Baselines fitted.')

## 5. Evaluation & Model Comparison

In [ ]:
import os
tft_ckpt_path = tft_ckpt if 'tft_ckpt' in dir() else None

from evaluation.evaluate import run_evaluation
comp_df, regime_df = run_evaluation(tft_ckpt=tft_ckpt_path)

In [ ]:
print('=== Model Comparison ===')
display(comp_df)
print()
print('=== Regime-Conditional QLIKE ===')
display(regime_df)

In [ ]:
from IPython.display import Image
Image('../outputs/forecast_comparison.png')

In [ ]:
import os
att_path = '../outputs/tft_attention_heatmap.png'
if os.path.exists(att_path):
    Image(att_path)
else:
    print('No attention heatmap found (TFT not trained or attention unavailable).')

## 6. Diebold-Mariano Pairwise Summary

In [ ]:
print('DM test: positive stat → row model worse than GARCH')
display(comp_df[['model', 'DM_vs_GARCH_stat', 'DM_p_value']])